# VirusTotal label check — are the dataset's labels right?

The Drebin-drift table's benign/malicious labels are verified against **VirusTotal**: for each SHA256, the number of engines flagging the file is fetched, and samples whose labels disagree with the engines are collected as suspect.

*Consolidated from `virustotal_com.ipynb` (originally in `mal_adv3`). **The original contained a real VirusTotal API key in plain text; it has been replaced with a placeholder** — put your own key in the marked cell before running. Outputs cleared.*

## Step 1 — Load the Drebin-drift table

In [ ]:
# https://drive.google.com/file/d/103hYqdHD6mjAUVqMAECgZEZ7U7YmSCP9/view?usp=sharing
import gdown
download_link = 'https://drive.google.com/uc?id=103hYqdHD6mjAUVqMAECgZEZ7U7YmSCP9'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
import pandas as pd
df = pd.read_parquet("/content/drebin_drift.parquet.zip")
df.head()

## Step 2 — Ask VirusTotal about one file

In [ ]:
import requests

# Replace with your own VirusTotal API key
api_key = "YOUR_VIRUSTOTAL_API_KEY"

# Replace '{id}' with SHA-256, SHA-1 or MD5 identifying the file
file_id = 'd7e1d0fb43c7d1d86d14a0e036539f5790656a8aa4842c8325c2b35790bf597f'

# Define the URL for the VirusTotal API
url = f'https://www.virustotal.com/api/v3/files/{file_id}'

# Set up the headers with your API key
headers = {
    'x-apikey': api_key
}

# Send a GET request to retrieve information about the file
response = requests.get(url, headers=headers)

# Access the 'last_analysis_results' field
analysis_results = response.json().get('data', {}).get('attributes', {}).get('last_analysis_results', {})

# Initialize a count for undetected results
detected_count = 0
undetected_count = 0
unsupported_count = 0

# Check each engine's result and count undetected results
for engine, result in analysis_results.items():
    if result['category'] == 'undetected':
        undetected_count += 1
    if result['category'] == 'malicious':
        detected_count += 1
    if result['category'] == "type-unsupported" :
        unsupported_count += 1


print('Probability of malware : ', detected_count/(detected_count + undetected_count + 0.0001))
print("Number of undetected results:", undetected_count)
print("Number of detected results:", detected_count)
print("Number of unsupported results:", unsupported_count)

In [ ]:
import requests
import time

def n_malicious(file_id):
    # Replace with your own VirusTotal API key
    api_key = "YOUR_VIRUSTOTAL_API_KEY"

    # Define the URL for the VirusTotal API
    url = f'https://www.virustotal.com/api/v3/files/{file_id}'

    # Set up the headers with your API key
    headers = {
        'x-apikey': api_key
    }

    retries = 10  # Number of retries
    delay = 1  # Delay between retries in seconds

    for attempt in range(retries):
        try:
            # Send a GET request to retrieve information about the file
            response = requests.get(url, headers=headers)
            response.raise_for_status()  # Raise an exception for HTTP errors

            # Access the 'last_analysis_results' field
            analysis_results = response.json().get('data', {}).get('attributes', {}).get('last_analysis_results', {})

            # Initialize a count for undetected results
            detected_count = 0

            # Check each engine's result and count undetected results
            for result in analysis_results.values():
                if result.get('category') == 'malicious':
                    detected_count += 1

            return detected_count
        except requests.RequestException as e:
            # Handle HTTP errors or other request-related exceptions
            print(f"An error occurred: {e}")
            if attempt < retries - 1:
                print(f"Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                print(f"Reached maximum retries. Stopping.")
                return None
        except Exception as e:
            # Handle any other unexpected exceptions
            print(f"An unexpected error occurred: {e}")
            return None

## Step 3 — Sweep the benign set for wrong labels

In [ ]:
df_benign = df[df['label']==0]
df_malicious = df[df['label']==1]

In [ ]:
from tqdm import tqdm

wrong_lables = []
for file_id in tqdm(df_benign['sha256'][:500]):
  count = n_malicious(file_id)
  if count > 5 :
      wrong_lables.append(file_id)
      print(file_id)
      print(count)

## Step 4 — Sweep the malicious set

In [ ]:
import requests

# Your VirusTotal API key
API_KEY = 'YOUR_VIRUSTOTAL_API_KEY"

# SHA256 hash of the file you want to check
file_sha256 = 'd7e1d0fb43c7d1d86d14a0e036539f5790656a8aa4842c8325c2b35790bf597f'

# URL for the VirusTotal API endpoint
url = f'https://www.virustotal.com/api/v3/files/{file_sha256}'

# Headers with the API key
headers = {
    'x-apikey': API_KEY
}

# Send a GET request to VirusTotal API
response = requests.get(url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Parse the JSON response
    result = response.json()

    # Check if the file is malicious
    if 'data' in result:
        data = result['data']
        if 'attributes' in data and 'last_analysis_stats' in data['attributes']:
            stats = data['attributes']['last_analysis_stats']
            if 'malicious' in stats and stats['malicious'] > 0:
                print("The file is malicious.")
            else:
                print("The file is not detected as malicious by any antivirus engines.")
        else:
            print("No analysis results available.")
    else:
        print("No data found for the file.")
else:
    print("Error:", response.status_code)